# BMIN 5200: Week 9 in-class exercise
## Entropy, information gain, and decision trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week09.ipynb)

**Time:** ~25 minutes · **Pairs with:** Information theory and machine learning

### Tasks
- Compute entropy, conditional entropy, and mutual information from the definitions on the slides, without a library
- Rank seven candidate predictors of 30-day readmission by how much information each provides about the outcome
- Fit a shallow decision tree that splits by information gain, and confirm that its first split is on the feature ranked highest by mutual information
- Read the tree as IF-THEN rules, and compare it with an unrestricted tree that memorizes the training set

### Background
A clinical risk model is built by some procedure for deciding which variable to examine first. Information theory measures that choice in bits, so a claim such as "hemoglobin is more informative than insurance type" becomes a number that can be checked. A shallow decision tree is one of the few learned models whose reasoning a clinician can read directly and dispute, which is one reason tree-structured rules remain common in clinical decision tools.

## Setup

No packages need to be installed; scikit-learn is preinstalled in Colab. The random seed is fixed so
that every laptop prints the same numbers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.feature_selection import mutual_info_classif

rng = np.random.default_rng(5200)
pd.set_option("display.width", 120)

## The cohort

The data are a synthetic 30-day readmission cohort of 1,200 fictional discharges; no real patient
data are used. Because the cohort is generated inline, the ground truth is known: readmission risk
depends on prior admissions, age, hemoglobin, and creatinine, and does not depend on distance from
the hospital.

Note the last lines of the generator, where `prior_admissions` is recorded incompletely for
patients whose care is split with a community partner network. This flaw is deliberate, it is
realistic, and Weeks 11 and 12 return to it. The same function with the same seed appears in the
Week 11 and Week 12 notebooks, so the model built today is the one you will explain and audit later
in the semester.

In [ ]:
def make_readmission_cohort(n_patients=1200, seed=5200):
    """Synthetic 30-day readmission cohort. Not real patient data.

    Identical in the Week 9, Week 11, and Week 12 notebooks — same code, same seed — so all
    three weeks work on exactly the same 1,200 discharges. Nothing is saved to disk between
    notebooks; each one regenerates the cohort inline. The function builds its own generator
    so the cohort does not depend on what else has drawn from `rng` above it.
    """
    rng = np.random.default_rng(seed)

    care_network = rng.choice(["in_network", "community_partner"],
                              size=n_patients, p=[0.68, 0.32])
    partner = care_network == "community_partner"

    age = np.clip(rng.normal(68, 12, n_patients), 40, 95).round(0)
    true_prior_admissions = rng.poisson(1.8, n_patients)
    hemoglobin = np.clip(rng.normal(12.4, 1.6, n_patients)
                         - 0.9 * (true_prior_admissions > 2), 7.0, 17.0)
    creatinine = np.clip(rng.lognormal(np.log(1.0), 0.32, n_patients), 0.4, 6.0)

    # Partner-network patients are likelier to be on Medicaid and to live farther out.
    ins_probs = np.where(partner[:, None],
                         np.array([0.18, 0.34, 0.48]),
                         np.array([0.52, 0.36, 0.12]))
    draw = rng.random(n_patients)
    insurance_type = np.array(["commercial", "medicare", "medicaid"])[
        (draw[:, None] > ins_probs.cumsum(axis=1)).sum(axis=1)]

    distance_from_hospital = np.round(
        np.clip(rng.gamma(2.0, np.where(partner, 9.0, 3.4)), 0.5, 90.0), 1)

    # Ground truth: risk is driven by the TRUE admission history, identically in both groups,
    # and never by distance from the hospital.
    log_odds = (-2.0
                + 0.62 * true_prior_admissions
                + 0.030 * (age - 68)
                - 0.26 * (hemoglobin - 12.4)
                + 0.55 * (creatinine - 1.0))
    risk = 1.0 / (1.0 + np.exp(-log_odds))
    readmitted_30d = (rng.random(n_patients) < risk).astype(int)

    # The planted flaw: only 45% of a partner-network patient's prior admissions reach our
    # chart, against 97% for patients who stay in network. The illness is the same; the
    # documentation is not.
    capture = np.where(partner, 0.45, 0.97)
    recorded_prior = rng.binomial(true_prior_admissions, capture)

    return pd.DataFrame({
        "age": age.astype(int),
        "prior_admissions": recorded_prior,
        "hemoglobin": hemoglobin.round(1),
        "creatinine": creatinine.round(2),
        "insurance_type": insurance_type,
        "distance_from_hospital": distance_from_hospital,
        "care_network": care_network,
        "readmitted_30d": readmitted_30d,
    })


cohort = make_readmission_cohort()
print(cohort.head(6).to_string(index=False))
print(f"\n{len(cohort)} discharges, {cohort.readmitted_30d.mean():.1%} readmitted within 30 days")

## Part 1: Entropy from the definition

Entropy is $H(X) = -\sum_x p(x)\log_2 p(x)$; with $\log_2$, the unit is bits. The "Interpretations
of H(X)" slide gives the reading that matters here: entropy is the minimum average number of bits
needed to encode the value of X, roughly the number of yes/no questions needed to identify it, so
it measures how much is not yet known. A fair coin has an entropy of 1 bit, and an outcome that
never varies has an entropy of 0 bits. Run the cell and check the three reference values before
the function is applied to the cohort.

In [ ]:
def entropy(labels):
    """H(X) in bits, straight from the definition. No library call."""
    labels = np.asarray(labels)
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / counts.sum()
    # abs() is here only so a degenerate distribution prints 0.0000 rather than -0.0000.
    return float(abs(-(probabilities * np.log2(probabilities)).sum()))


print(f"fair coin              H = {entropy([0, 1]):.4f} bits")
print(f"nothing ever happens   H = {entropy([0, 0, 0, 0]):.4f} bits")
print(f"1-in-4 event           H = {entropy([0, 0, 0, 1]):.4f} bits")

outcome = cohort["readmitted_30d"]
h_outcome = entropy(outcome)
print(f"\n30-day readmission     H = {h_outcome:.4f} bits "
      f"(base rate {outcome.mean():.1%})")

Before any feature is examined, the outcome carries 0.92 bits of uncertainty. A predictor is worth
the number of those bits it removes. That quantity is the mutual information, which Part 2 computes.

## Part 2: Conditional entropy and mutual information

Conditional entropy is the weighted average of the outcome's entropy within each group of patients
defined by a feature: $H(Y \mid X) = \sum_x p(x)\,H(Y \mid X{=}x)$ (the "Conditional Entropy"
slide). The weighting matters: a group of four patients should count for less than a group of four
hundred. Mutual information is the difference $I(X;Y) = H(Y) - H(Y \mid X)$, the bits of
uncertainty about the outcome that the feature removes (the "Mutual Information" slide).

These definitions apply to discrete variables, so `bin_feature` converts each feature with more
than six distinct values into quartiles. This includes `prior_admissions`; because most patients
have zero or one prior admission, its quartile edges collapse to three groups (0–1, 2, and 3 or
more). Complete the line marked TODO. The placeholder ignores the feature, so every mutual
information value is reported as 0.

In [ ]:
def bin_feature(column, n_bins=4):
    """Entropy is defined over discrete values, so features with many values become quartiles."""
    if column.dtype == object or column.nunique() <= 6:
        return column.astype(str)
    return pd.qcut(column, n_bins, labels=False, duplicates="drop").astype(str)


def conditional_entropy(feature, labels):
    """H(Y | X): the outcome entropy that survives after you know the feature."""
    feature, labels = np.asarray(feature), np.asarray(labels)
    total = len(labels)
    running = 0.0
    for value in np.unique(feature):
        in_bin = feature == value
        weight = in_bin.sum() / total
        # TODO: add this bin's contribution, weight * H(Y | X = value), to `running`.
        # The placeholder uses every patient's outcome, so the feature appears to tell us
        # nothing and every I(X;Y) is 0.
        running += weight * entropy(labels)
    return running


candidates = ["age", "prior_admissions", "hemoglobin", "creatinine",
              "insurance_type", "distance_from_hospital", "care_network"]

ranking = []
for feature_name in candidates:
    binned = bin_feature(cohort[feature_name])
    h_given = conditional_entropy(binned, outcome)
    ranking.append({"feature": feature_name,
                    "H(Y|X)": round(h_given, 4),
                    "I(X;Y) bits": round(h_outcome - h_given, 4)})

ranking = pd.DataFrame(ranking).sort_values("I(X;Y) bits", ascending=False)
print(f"H(Y) = {h_outcome:.4f} bits before we know anything\n")
print(ranking.to_string(index=False))

Check your work against one number. With `conditional_entropy` completed correctly,
`prior_admissions` leaves 0.8393 bits of uncertainty, a mutual information of 0.0814 bits. If the
next cell reports "not yet", check that each bin's entropy is computed from the patients in that
bin only, and that it is weighted by the bin's share of patients.

In [ ]:
check = conditional_entropy(bin_feature(cohort["prior_admissions"]), outcome)
if abs(check - 0.8393) < 5e-4:
    print(f"H(Y | prior_admissions) = {check:.4f} bits — matches, carry on")
else:
    print(f"H(Y | prior_admissions) = {check:.4f} bits — not yet; expected 0.8393")
    print("Compute each bin's entropy from that bin's patients, weighted by the bin's size.")

Two points about this ranking. First, the values are small: the best single predictor removes less
than a tenth of a bit, which reflects how difficult readmission is to predict from a few discharge
variables. Second, `distance_from_hospital` is near the bottom, which is correct, because the
generator gives it no effect on the outcome. In Week 11, a random forest's built-in feature
importance will rank it above `prior_admissions`.

## Part 3: Decision tree induction

The "Decision Trees: Training the model" slide lists several split criteria. scikit-learn's
`DecisionTreeClassifier` implements CART and uses the Gini index by default; setting
`criterion="entropy"` makes it choose each split by information gain, as ID3 does. The tree is
limited to depth 3 so that it can be read in full. 30% of the cohort is held out for testing,
following the "Data preparation: A split-sample method" slide, and Part 4 uses this test set.
`export_text` prints the induced tree.

### Predictions

The tree is built greedily: at each node it takes the split with the largest information gain. Its
first split is therefore chosen using the quantity computed in Part 2. Before running the next cell,
agree as a class on answers to the following:

1. Which feature will be at the root of the tree?
2. What threshold will the root split use?

In [ ]:
encoded = cohort[candidates].copy()
encoded["insurance_type"] = encoded["insurance_type"].map(
    {"commercial": 0, "medicare": 1, "medicaid": 2})
encoded["care_network"] = encoded["care_network"].map(
    {"in_network": 0, "community_partner": 1})

X_train, X_test, y_train, y_test = train_test_split(
    encoded, outcome, test_size=0.3, random_state=5200, stratify=outcome)

shallow_tree = DecisionTreeClassifier(max_depth=3, criterion="entropy", random_state=5200)
shallow_tree.fit(X_train, y_train)

root_feature = candidates[shallow_tree.tree_.feature[0]]
root_threshold = shallow_tree.tree_.threshold[0]
print(f"Root split: {root_feature} <= {root_threshold:.2f}\n")
print(export_text(shallow_tree, feature_names=candidates, decimals=1))

The root split is on `prior_admissions`, the feature ranked first by mutual information. This is
expected: the information gain of a split is the mutual information between the outcome and the
split, computed for a two-way split of the training set rather than for quartile bins of the full
cohort, which is why the values differ. The next cell compares both results with scikit-learn's
own mutual information estimator, which uses a nearest-neighbor method for continuous features
instead of bins.

In [ ]:
library_mi = pd.Series(
    mutual_info_classif(encoded, outcome, random_state=5200,
                        discrete_features=[c in ("prior_admissions", "insurance_type",
                                                 "care_network") for c in candidates]),
    index=candidates).sort_values(ascending=False)

print("scikit-learn mutual_info_classif, in nats rather than bits; compare the order:")
print(library_mi.round(4).to_string())
hand_rolled_winner = (ranking.iloc[0]["feature"] if ranking["I(X;Y) bits"].max() > 0
                      else "(fill in the Part 2 TODO first)")
print(f"\nHand-rolled winner : {hand_rolled_winner}")
print(f"sklearn winner     : {library_mi.index[0]}")
print(f"Tree's root split  : {root_feature}")

Each root-to-leaf path is a conjunction of tests that ends in a conclusion. This is an IF-THEN
production rule of the kind written by hand in Weeks 6 and 7. The difference is where the rules
came from: in those weeks they were written from clinical knowledge, and here they were induced from
840 training discharges. The points the Week 6 notebook made about conflict resolution and about
explaining a conclusion apply to these rules as well.

In [ ]:
def tree_to_rules(tree, feature_names):
    """Walk each root-to-leaf path and emit it as an IF/THEN rule, Week 6 style."""
    structure = tree.tree_
    rules = []

    def walk(node, conditions):
        if structure.feature[node] == -2:                     # leaf
            counts = structure.value[node][0]
            predicted = int(np.argmax(counts))
            support = int(structure.n_node_samples[node])
            confidence = counts[predicted] / counts.sum()
            rules.append((conditions, predicted, support, confidence))
            return
        name = feature_names[structure.feature[node]]
        cut = structure.threshold[node]
        walk(structure.children_left[node], conditions + [f"{name} <= {cut:.1f}"])
        walk(structure.children_right[node], conditions + [f"{name} > {cut:.1f}"])

    walk(0, [])
    return rules


for conditions, predicted, support, confidence in tree_to_rules(shallow_tree, candidates):
    verdict = "readmit within 30d" if predicted == 1 else "no readmission"
    print(f"IF   {' AND '.join(conditions)}")
    print(f"THEN {verdict}   (n={support}, confidence {confidence:.2f})\n")

One of these rules reads: IF `prior_admissions` > 2.5 AND `age` ≤ 70.5 AND
`distance_from_hospital` > 12.2 THEN readmit, with a confidence of 0.91. The generator gives
distance no effect on readmission, so this rule is an artifact of the 11 training patients it rests
on. This is the problem described on the "Tree Induction" slide: it may be coincidence that a
condition holds in all the examples. Rules deep in a tree are supported by few patients, and their
confidence should be read together with the support (n) printed beside it.

## Part 4: Unrestricted tree growth

Nothing in the induction algorithm requires stopping at depth 3. Without a depth limit, the
algorithm keeps splitting until every leaf is pure, because on any finite sample there is almost
always a threshold that separates the remaining patients who disagree. The next cell fits an
unrestricted tree and compares training and test accuracy for the two trees.

### Predictions

1. What will the unrestricted tree's training accuracy be?
2. Will its test accuracy be higher or lower than that of the depth-3 tree?

In [ ]:
deep_tree = DecisionTreeClassifier(criterion="entropy", random_state=5200)  # no max_depth
deep_tree.fit(X_train, y_train)

def score_tree(name, tree):
    return {
        "tree": name,
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "train acc": round(accuracy_score(y_train, tree.predict(X_train)), 3),
        "test acc": round(accuracy_score(y_test, tree.predict(X_test)), 3),
        "test AUC": round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3),
    }

print(pd.DataFrame([score_tree("depth 3", shallow_tree),
                    score_tree("unrestricted", deep_tree)]).to_string(index=False))

The unrestricted tree classifies every training patient correctly and performs worse than the
depth-3 tree on the test set. It has 200 leaves, more than half of which contain one or two training
patients, so it has memorized the training set rather than learned a pattern that generalizes. This
is the argument behind the split-sample and cross-validation slides: training accuracy is not
evidence of performance, because a model with enough capacity can always drive it to 1.0. Complete
the TODO below to find the depth at which cross-validated performance stops improving.

In [ ]:
depths_to_try = [1, 2, 3, 4, 6, 8, 12, None]
cv_results = []
for depth in depths_to_try:
    candidate_tree = DecisionTreeClassifier(max_depth=depth, criterion="entropy",
                                            random_state=5200)
    # TODO: replace the training-set score below with 5-fold cross-validated AUC, per the
    # "K-fold cross validation" slide:
    #     scores = cross_val_score(candidate_tree, encoded, outcome, cv=5, scoring="roc_auc")
    #     auc = scores.mean()
    candidate_tree.fit(encoded, outcome)
    auc = roc_auc_score(outcome, candidate_tree.predict_proba(encoded)[:, 1])
    cv_results.append({"max_depth": depth if depth else "none",
                       "AUC": round(auc, 3)})

print(pd.DataFrame(cv_results).to_string(index=False))
print("\nIf the AUC rises to 1.0 and stays there, the tree is still being scored on the")
print("patients it was trained on.")

For simplicity, the cell above cross-validates on the full cohort. In practice the depth would be
chosen by cross-validation within the training set, and the test set would be used once, at the
end (the "Training / Testing / Validation" slide).

The last cell plots the ROC curve for each tree on the held-out 30%, as on the "ROC Curve and AUC"
slide; the diagonal corresponds to random guessing. The unrestricted tree's curve lies close to the
diagonal. Its leaves are pure, so it gives every test patient a predicted risk of exactly 0 or 1,
which provides almost no ranking of patients by risk.

In [ ]:
plt.figure(figsize=(5, 5))
for name, tree in [("depth 3", shallow_tree), ("unrestricted", deep_tree)]:
    fpr, tpr, _ = roc_curve(y_test, tree.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Held-out ROC, 30-day readmission")
plt.legend(loc="lower right")
plt.show()

## Discussion

1. The best single feature removes 0.08 of the 0.92 bits of uncertainty about readmission. If you
   were asked to approve this model for assigning transitional-care nurses, what else would you
   need to know to decide whether that is enough?

2. The depth-3 tree is a rule base that can be read, and the unrestricted tree is not. At what depth
   does a tree stop being explainable? Is the relevant limit the depth, or whether each rule has
   enough patients behind it to be trusted?

3. Mutual information ranked `prior_admissions` first, but this cohort records prior admissions
   less completely for one group of patients. Can mutual information detect that a feature is
   measured unevenly across groups? If not, what could?

## Carried forward to Weeks 11 and 12

The cohort returns twice. In Week 11 you will train a random forest on the same data and use SHAP
and LIME to explain its predictions. In Week 12 you will audit that model and find the recording gap
planted in `capture`, as a measurable difference in which patients are flagged for follow-up.

## Solutions

Completed versions of the two TODOs, in a markdown cell so they are not executed.

**Part 2: `conditional_entropy`**

```python
def conditional_entropy(feature, labels):
    feature, labels = np.asarray(feature), np.asarray(labels)
    total = len(labels)
    running = 0.0
    for value in np.unique(feature):
        in_bin = feature == value
        weight = in_bin.sum() / total
        running += weight * entropy(labels[in_bin])
    return running
```

The fix is `labels[in_bin]` in place of `labels`: compute the outcome's entropy within the patients
in each bin, then average over the bins, weighted by size.

**Part 4: cross-validated AUC**

```python
depths_to_try = [1, 2, 3, 4, 6, 8, 12, None]
cv_results = []
for depth in depths_to_try:
    candidate_tree = DecisionTreeClassifier(max_depth=depth, criterion="entropy",
                                            random_state=5200)
    scores = cross_val_score(candidate_tree, encoded, outcome, cv=5, scoring="roc_auc")
    cv_results.append({"max_depth": depth if depth else "none",
                       "AUC": round(scores.mean(), 3)})

print(pd.DataFrame(cv_results).to_string(index=False))
```

Cross-validated AUC peaks at depth 4 (0.680) and falls to 0.567 for the unrestricted tree. The
training-set version reaches 1.000, which is why it should never be reported as a measure of
performance.